In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df_model = pd.read_csv('/content/drive/MyDrive/Dissertation_Project/final_model_dataset.csv')

df_model.head()

In [ ]:
print("Rows and columns:", df_model.shape)
print(df_model.columns)

In [ ]:
print("Rows and columns:", df_model.shape)
print(df_model.columns)

In [ ]:
df_model.info()

In [ ]:
df_model.isnull().sum()

In [ ]:
df_model.isnull().sum()

In [ ]:
df_model.duplicated().sum()

In [ ]:
df_model[['yield', 'temperature', 'precipitation', 'solar_radiation', 'dry_matter']].describe()

In [ ]:
df_model['fertilizer_code'].value_counts()

In [ ]:
df_model.to_csv('/content/drive/MyDrive/Dissertation_Project/final_model_dataset.csv', index=False)

In [ ]:
df_model[['yield', 'temperature', 'precipitation', 'solar_radiation', 'dry_matter']].describe()

In [ ]:
df_model['fertilizer_code'].value_counts()

In [ ]:
df_yearly = df_model.groupby('year')['yield'].mean().reset_index()

plt.figure(figsize=(12,5))
plt.plot(df_yearly['year'], df_yearly['yield'], marker='o')
plt.xticks(df_yearly['year'], rotation=45)
plt.xlabel('Year')
plt.ylabel('Average Yield')
plt.title('Average Maize Yield Trend in the UK (1997–2017)')
plt.grid()
plt.show()

In [ ]:
df_yearly = df_model.groupby('year')['yield'].mean().reset_index()

# Ensure all years are present (fills missing years if any)
all_years = pd.DataFrame({'year': range(1997, 2018)})
df_yearly = all_years.merge(df_yearly, on='year', how='left')

plt.figure(figsize=(12,5))
plt.plot(df_yearly['year'], df_yearly['yield'], marker='o')

# Force equal spacing
plt.xticks(range(1997, 2018), rotation=45)

plt.xlabel('Year')
plt.ylabel('Average Yield')
plt.title('Average Maize Yield Trend in the UK (1997–2017)')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(df_yearly['year'], df_yearly['yield'], marker='o')

plt.xticks(range(1997, 2018), rotation=45)  # show all years

plt.xlabel('Year')
plt.ylabel('Average Yield')
plt.title('Average Maize Yield Trend in the UK (1997–2017)')

plt.grid()
plt.show()

In [ ]:
corr = df_model[['yield', 'temperature', 'precipitation', 'solar_radiation', 'dry_matter']].corr()
print(corr)

In [ ]:
corr = df_model[['yield', 'temperature', 'precipitation', 'solar_radiation', 'dry_matter']].corr()
print(corr)

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Between Variables')
plt.show()

In [ ]:
df_fert = df_model.groupby('fertilizer_code')['yield'].mean().sort_values(ascending=False)

print(df_fert)

In [ ]:
df_fert.plot(kind='bar', figsize=(10,5))
plt.title('Average Yield by Fertilizer Type')
plt.xlabel('Fertilizer')
plt.ylabel('Average Yield')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_fert = df_model.groupby('fertilizer_code')['yield'].mean().sort_values(ascending=False)

top10 = df_fert.head(10)

plt.figure(figsize=(10,5))
top10.plot(kind='bar')

plt.title('Top 10 Fertilizer Treatments by Average Yield')
plt.xlabel('Fertilizer Type')
plt.ylabel('Average Yield')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
df_fert.head(15).plot(kind='barh')

plt.title('Top Fertilizer Treatments by Yield')
plt.xlabel('Average Yield')
plt.ylabel('Fertilizer')
plt.tight_layout()
plt.show()

In [ ]:
def simplify_fertilizer(x):
    if 'FYM' in x:
        return 'Organic (FYM)'
    elif 'N' in x:
        return 'Nitrogen-based'
    elif 'PK' in x:
        return 'PK-based'
    else:
        return 'Other'

df_model['fertilizer_group'] = df_model['fertilizer_code'].apply(simplify_fertilizer)

df_group = df_model.groupby('fertilizer_group')['yield'].mean()

df_group.plot(kind='bar')
plt.title('Yield by Fertilizer Group')
plt.show()

In [ ]:
plt.hist(df_model['yield'], bins=20)
plt.title('Distribution of Maize Yield')
plt.xlabel('Yield')
plt.ylabel('Frequency')
plt.show()

In [ ]:
X = pd.get_dummies(
    df_model[['temperature', 'precipitation', 'solar_radiation', 'dry_matter', 'fertilizer_code']]
)

y = df_model['yield']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression R²:", r2_score(y_test, y_pred_lr))
print("Linear Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))

In [ ]:
print("Linear Regression R²:", r2_score(y_test, y_pred_lr))
print("Linear Regression RMSE:", ...)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest R²:", r2_score(y_test, y_pred_rf))
print("Random Forest RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))

In [ ]:
from xgboost import XGBRegressor

# Fix column names for XGBoost
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

X_train_xgb.columns = X_train_xgb.columns.astype(str).str.replace('[', '', regex=False)\
                                             .str.replace(']', '', regex=False)\
                                             .str.replace('<', '', regex=False)

X_test_xgb.columns = X_test_xgb.columns.astype(str).str.replace('[', '', regex=False)\
                                           .str.replace(']', '', regex=False)\
                                           .str.replace('<', '', regex=False)

xgb = XGBRegressor(random_state=42)
xgb.fit(X_train_xgb, y_train)

y_pred_xgb = xgb.predict(X_test_xgb)

print("XGBoost R²:", r2_score(y_test, y_pred_xgb))
print("XGBoost RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

svr.fit(X_train, y_train)

y_pred_svr = svr.predict(X_test)

print("SVR R²:", r2_score(y_test, y_pred_svr))
print("SVR RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_svr)))

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost', 'SVR'],
    'R²': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_xgb),
        r2_score(y_test, y_pred_svr)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        np.sqrt(mean_squared_error(y_test, y_pred_svr))
    ]
})

results

In [ ]:
!pip install shap

In [ ]:
import shap

explainer = shap.Explainer(xgb, X_train_xgb)
shap_values = explainer(X_test_xgb)

In [ ]:
# Convert all XGBoost input columns to numeric float
X_train_shap = X_train_xgb.copy().astype(float)
X_test_shap = X_test_xgb.copy().astype(float)

# Refit XGBoost using numeric data
xgb.fit(X_train_shap, y_train)

# Run SHAP
import shap

explainer = shap.Explainer(xgb, X_train_shap)
shap_values = explainer(X_test_shap)

In [ ]:
shap.summary_plot(shap_values, X_test_shap)

In [ ]:
print(X_train_shap.dtypes.unique())

In [ ]:
print("XGBoost Training R²:", r2_score(y_train, xgb.predict(X_train_shap)))
print("XGBoost Testing R²:", r2_score(y_test, xgb.predict(X_test_shap)))

print("XGBoost Training RMSE:", np.sqrt(mean_squared_error(y_train, xgb.predict(X_train_shap))))
print("XGBoost Testing RMSE:", np.sqrt(mean_squared_error(y_test, xgb.predict(X_test_shap))))

In [ ]:
!pip install lime

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    training_data=X_train_xgb.values,
    feature_names=X_train_xgb.columns,
    mode='regression'
)

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=X_train_shap.values,
    feature_names=X_train_shap.columns.tolist(),
    mode='regression'
)

In [ ]:
i = 0  # first test example

lime_exp = lime_explainer.explain_instance(
    X_test_shap.iloc[i].values,
    xgb.predict,
    num_features=6
)

lime_exp.show_in_notebook(show_table=True)

In [ ]:
lime_exp.as_list()

In [ ]:
row = X_test_shap.iloc[i]

used_fertilizer = row[row == 1]

print(used_fertilizer)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df_model = pd.read_csv('/content/drive/MyDrive/Dissertation_Project/final_model_dataset.csv')

df_model.head()

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/final_dataset.csv')

df.head()